# Import Librarys and Files

## Imports

### Add top level to the import path

In [1]:
import sys
import os

# Add the project root to sys.path
project_root = os.path.abspath("../..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

### Import some classes/modules we'll be using

In [2]:
import ROOT as r
print("ROOT VERSION:")
print(r.gROOT.GetVersion())
import matplotlib.pyplot as plt
import numpy as np
from importlib import reload
from tqdm.notebook import tqdm 

import deprecated.utils.utils as utils
import deprecated.utils.plotting_utils as plotting_utils
from deprecated.models.event_patterns import EventPatterns
from models.pattern import Pattern
from models.vertex import Vertex
from models.tracklet import Tracklet
from models.hit import Hit

ROOT VERSION:
6.32.02


## Load Shared libraries to access methods of ROOT objects in python

In [3]:
PI_ROOT_DICT_PATH = "/home/jack/large_projects/PIONEER/install/lib/libPiRootDict.so" #Change to match system path
r.gSystem.Load(PI_ROOT_DICT_PATH)

0

## Load Pattern finding truth data
This also contains what we need to reconstruct patterns. So it gives validation (true patterns) and a method to create our own pattern

In [4]:
rec_file_path = "/home/jack/large_projects/simulation/workspace/playground/reco_algorithm_tests/data/test_vertices5.root"
#truth_rec_file_path = "/home/jack/large_projects/simulation/workspace/playground/reco_algorithm_tests/data/test_new_imp_truth.root"

# Print creation times
utils.print_file_creation_time(rec_file_path)
#utils.print_file_creation_time(truth_rec_file_path)

# Open the files
rec_file = r.TFile(rec_file_path, "READ")
#truth_rec_file = r.TFile(truth_rec_file_path, "READ")


The file '/home/jack/large_projects/simulation/workspace/playground/reco_algorithm_tests/data/test_vertices5.root' was created on: 2025-07-09 19:58:40


## Get ROOT objects from our file
Making these more than once can cause memory leaks

In [7]:
tree = rec_file.Get("rec")
#truth_tree = truth_rec_file.Get("rec")
#geoHelper = rec_file.Get("GeoHeader")

## (Optional) Validate the file

In [9]:
tree.GetEntry(0)  # Load the first event

for summary in getattr(tree, "summaryVec"):
    patterns = summary.GetPattern()
    for pattern in patterns:
        vertices = pattern.GetVertices()
        for i in range(vertices.GetEntries()):
            vertex = vertices.At(i)
            tracklets = vertex.GetTracklets()
            for j in range(tracklets.GetEntries()):
                tracklet = tracklets.At(j)
                print(f"Tracklet ID: {tracklet.GetPID()},")


AttributeError: 'TObject' object has no attribute 'GetPattern'

Error in <TInterpreter::AutoParse>: Error parsing payload code for class PIRECSummary with content:

#line 1 "libPiRootDict dictionary payload"

#ifndef __ROOFIT_NOBANNER
  #define __ROOFIT_NOBANNER 1
#endif
#ifndef __ROOFIT_NOBANNER
  #define __ROOFIT_NOBANNER 1
#endif
#ifndef VECCORE_ENABLE_VC
  #define VECCORE_ENABLE_VC 1
#endif

#define _BACKWARD_BACKWARD_WARNING_H
// Inline headers
#include "/home/jack/large_projects/PIONEER/shared/cuts/include/PIRECCut.hh"
#include "/home/jack/large_projects/PIONEER/shared/cuts/include/PIRECCutAtarBox.hh"
#include "/home/jack/large_projects/PIONEER/shared/cuts/include/PIRECCutAtarHits.hh"
#include "/home/jack/large_projects/PIONEER/shared/cuts/include/PIRECCutAtarKink.hh"
#include "/home/jack/large_projects/PIONEER/shared/cuts/include/PIRECCutBhabha.hh"
#include "/home/jack/large_projects/PIONEER/shared/cuts/include/PIRECCutDelayedDEDZ.hh"
#include "/home/jack/large_projects/PIONEER/shared/cuts/include/PIRECCutDelayedE5.hh"
#include "/home/jack/l

# Create Python Structure from Event Data

## Initialize Event Patterns Object

In [ ]:
# Initialize Event patterns without tracklet, vertex, and pattern forming algorithms (we'll add them later)
event_index = 0
event_patterns = EventPatterns(event_index, None, None, None, validator = None)

# For "normal use" you'd just initialize with all 3 pattern forming algorithms here, then call event_patterns.form_all(file, index)
# Below, we go through each step instead:


## Create tracklets for an event

In [ ]:
# Choose a tracklet forming algorithm, I'm use this default one, but you can write your own child class of TrackletFormer
from algorithms.tracklet.known_patterns_tracklet_former import KnownPatternsTrackletFormer
event_patterns.tracklet_former = KnownPatternsTrackletFormer(truth_tree)
event_patterns.form_tracklets(tree, geoHelper, event_index)

# Plot the tracklets
reload(plotting_utils)
plotting_utils.plot_event(event_patterns)

unique_tracklets = {
    tracklet
    for pattern in event_patterns.get_patterns()
    for vertex in pattern.get_vertices()
    for tracklet in vertex.get_tracklets()
}
print(f"Total unique tracklets: {len(unique_tracklets)}")


# See the extra info stored at this point
print("Extra event info:")
print(event_patterns.extra_info)

## Create Vertices from Tracklets

In [ ]:
# Choose a vertex forming algorithm, I'm using this kmeans one, but you can write your own child class of VertexFormer
from algorithms.vertex.known_patterns_vertex_former import KnownPatternsVertexFormer
event_patterns.vertex_former = KnownPatternsVertexFormer()

event_patterns.form_vertices()
print(event_patterns.patterns)

reload(plotting_utils)
plotting_utils.plot_event(event_patterns)

# See the extra info stored at this point
print("Extra event info:")
print(event_patterns.extra_info)

## Create Patterns from Vertices

In [ ]:
# Choose a pattern forming algorithm, I'm using this default one, but you can write your own child class of PatternFormer
from algorithms.pattern.known_patterns_pattern_former import KnownPatternsPatternFormer
event_patterns.pattern_former = KnownPatternsPatternFormer()
event_patterns.form_patterns()

# Print patterns and the unique tracklets within each pattern along with vertex information
patterns = event_patterns.get_patterns()
for pattern in patterns:
    print(f"\nPattern with {len(pattern.get_vertices())} vertices.")

    # Print info for each vertex in the pattern
    for vertex in pattern.get_vertices():
        tracklet_ids = [t.tracklet_id for t in vertex.get_tracklets()]
        print(f"  Vertex {vertex.vertex_id} (tracklet_ids={tracklet_ids})")

    print("  Unique Tracklets:")
    unique_tracklets = pattern.get_unique_tracklets()
    for tracklet in unique_tracklets:
        print(f"    {tracklet}")


# See the extra info stored at this point
print()
print("Extra event info:")
print(event_patterns.extra_info)

## (Optional) Validate Event Against Truth
This requires some truth information to be stored in the pattern's "extra_info". In this case, we stored it in the tracklet forming stage

In [ ]:
# Choose a validation algorithm, I'm using one that checks that each pattern in the reco and truth contain the same tracklet ids
# but you can write your own child class of EventValidator
from deprecated.algorithms.validation.tracklet_grouping_validator import TrackletGroupingValidator
event_patterns.validator = TrackletGroupingValidator(verbose = 1)
event_patterns.validate()


# Reconstruct Multiple Events

## Make cuts on the data set

In [ ]:
# Number of events to process (set to None or a large number to process all events)
MAX_EVENTS = None
N_EVENTS = tree.GetEntries() if MAX_EVENTS is None else min(MAX_EVENTS, tree_pf.GetEntries())

use_pitar = True
use_in_fid_vol = True

nentries = tree.GetEntries()
all_events = []  # Store event IDs of all events that pass the initial cuts

for iEntry in tqdm(range(min(nentries, N_EVENTS)), desc="Processing Events"):

    patternFailed = False
    tree.GetEntry(iEntry)

    # SELECTIONS
    pitar = any(info.Has(r.kPitar) for info in tree.infoVec)

    if (not pitar) and use_pitar:
        continue

    in_fid_vol = True
    for tracklet in tree.trackletVec:
        for hit in tracklet.GetAllHits():
            vname = geoHelper.GetVolumeName(hit.GetVID()).Data()
            if 'atar' in vname:
                side = vname[11]
                if (side == "f" and abs(geoHelper.GetX(hit.GetVID())) > 8) or \
                   (side == "b" and abs(geoHelper.GetY(hit.GetVID())) > 8):
                    in_fid_vol = False

    if (not in_fid_vol) and use_in_fid_vol:
        continue

    all_events.append(iEntry)

## Run Reconstruction on Events

### (Optional) Reload algorithms

In [ ]:
import importlib

# Reload tracklet formers
import algorithms.tracklet.known_patterns_tracklet_former
importlib.reload(algorithms.tracklet.known_patterns_tracklet_former)

# Reload vertex formers
import algorithms.vertex.known_patterns_vertex_former
importlib.reload(algorithms.vertex.known_patterns_vertex_former)


In [ ]:
from algorithms.tracklet.known_patterns_tracklet_former import KnownPatternsTrackletFormer

from algorithms.vertex.known_patterns_vertex_former import KnownPatternsVertexFormer

from algorithms.pattern.known_patterns_pattern_former import KnownPatternsPatternFormer

from deprecated.algorithms.validation.tracklet_grouping_validator import TrackletGroupingValidator

# Preallocate array
reconstructed_events = np.empty(len(all_events), dtype=object)

# tqdm with simple progress description
with tqdm(total=len(all_events), desc="Reconstructing events") as pbar:
    for i, event_index in enumerate(all_events):
        event_patterns = EventPatterns(event_index, 
                                       KnownPatternsTrackletFormer(truth_tree), 
                                       KnownPatternsVertexFormer(),
                                       KnownPatternsPatternFormer(), 
                                       TrackletGroupingValidator())
        event_patterns.form_all(tree, geoHelper, event_index)
        reconstructed_events[i] = event_patterns
        pbar.update(1)


# Check/View Algorithm Performance

## Performance vs. Particle composition

### Performance categorized by reconstructed particle componsition
Sometimes tracklets exist in truth, but do not create a hit in the atar. As a result, we cannot always construct the true particle composition.

In [ ]:
reload(plotting_utils)
title = f"Pattern Reconstruction by Reconstructed Particle Composition ($N_{{\\text{{events}}}} = {len(reconstructed_events)}$)"
plotting_utils.plot_event_classification_from_patterns(reconstructed_events, use_truth_particles = False, title =  title)

### Performance categorized by true particle componsition
We can tag each event with the true particle composition information and plot against that instead to also see performance based on true particle composition

In [ ]:
reload(plotting_utils)
title = f"Pattern Reconstruction by True Particle Composition ($N_{{\\text{{events}}}} = {len(reconstructed_events)}$)"
plotting_utils.plot_event_classification_from_patterns(reconstructed_events, use_truth_particles = True, title =  title)

## Performance vs. True Number of Patterns

In [ ]:
reload(plotting_utils)
title = f"Pattern Reconstruction by True Particle Composition ($N_{{\\text{{events}}}} = {len(reconstructed_events)}$)"
plotting_utils.plot_event_patterns_summary(reconstructed_events, title =  None)